In [99]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV


import warnings
warnings.filterwarnings("ignore")

In [100]:
def load_and_explore_dataset(filepath):
    print("=" * 60)
    print("LOAD AND  EXPLORE DATASET")
    print("=" * 60)

    df = pd.read_csv(filepath)

    print("Shape of the dataset")
    print(df.shape)
    print("\nCheck for missing values")
    print(df.isnull().sum())
    print("\nFirst five rows:")
    print(df.head())
    print("\nDescriptive stats")
    print(df.describe())
    print("\nDataset Info")
    print(df.info())
    print("\nState Distribution:")
    print(df["state"].value_counts())
    print("\nSmoker Distribution:")
    print(df["smoker"].value_counts())

    return df

In [101]:
def features_data(df):
    print("\n" + "=" * 60)
    print("FEATURES DATA")
    print("=" * 60)
    
    numerical_features = ["age", "bmi", "children"]
    categorical_features = ["gender", "smoker", "state"]

    target_column = "hospital_bill"

    X = df[numerical_features + categorical_features]
    y = df[target_column]

    feature_columns = numerical_features + categorical_features

    print("Features Shape:", X.shape)
    print("Target Shape:", y.shape)

    return X, y, numerical_features, categorical_features, feature_columns

In [102]:
def split_data(X, y, test_size=0.2, random_state=42):
    print("\n" + "=" * 60)
    print("SPLITING DATA")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining set size", X_train.shape[0])
    print("Testing set size", X_test.shape[0])
    print("\nTraining set price range {:.2f} - {:.2f}".format(
        float(y_train.min()), float(y_train.max())
    ))
    print("\nTesting set price range {:.2f} - {:.2f}".format(
        float(y_test.min()), float(y_test.max())
    ))

    return X_train, X_test, y_train, y_test

In [103]:
# def build_and_train_model(X_train, y_train, numerical_features, categorical_features):

#     print("\n" + "=" * 60)
#     print("BUILDING & TRAINING PIPELINE WITH GRID SEARCH")
#     print("=" * 60)

#     # transformers
#     numerical_transformer = Pipeline(steps=[
#         ("imputer", SimpleImputer(strategy="median")) 
#     ])
    
#     categorical_transformer = Pipeline(steps=[
#         ("imputer", SimpleImputer(strategy="most_frequent")),
#         ("encoder", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
#     ])

#     preprocessor = ColumnTransformer(
#         transformers=[
#             ("num", numerical_transformer, numerical_features),
#             ("cat", categorical_transformer, categorical_features)
#         ])

#     # base pipeline
#     model = Pipeline(
#         steps=[
#             ("preprocessor", preprocessor),
#             ("randomforestregressor", RandomForestRegressor(
#                 random_state=42,
#                 n_jobs=1))
#         ]
#     )

#     # parameter grid
#     param_grid = {
#         "randomforestregressor__n_estimators": [300, 500, 1000],
#         "randomforestregressor__max_depth": [3, 5, 7],
#         "randomforestregressor__min_samples_split": [2, 5, 10],
#         "randomforestregressor__min_samples_leaf": [2, 5, 10],
#         # "randomforestregressor__max_features": ["sqrt", "log2"]
#     }

#     # grid search
#     grid_search = GridSearchCV(model, param_grid, cv=5, scoring="r2", n_jobs=-1, verbose=1)

#     y_train_log = np.log1p(y_train)
#     # y_test_log  = np.log1p(y_test)   # only for evaluation, never fed to model
   
#     grid_search.fit(X_train, y_train_log)

#     print("\nBest Parameters Found:")
#     print(grid_search.best_params_)

#     print("\nBest Cross-Validation R2 Score:")
#     print(grid_search.best_score_)

#     return grid_search.best_estimator_


In [104]:
def build_and_train_model(X_train, y_train, numerical_features, categorical_features):
    print("\n" + "=" * 60)
    print("BUILDING & TRAINING PIPELINE")
    print("=" * 60)

    numerical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
    
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numerical_transformer, numerical_features),
            ("cat", categorical_transformer, categorical_features)
        ])

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("randomforestregressor", RandomForestRegressor( n_estimators=300, max_depth=8, min_samples_split=10,
        min_samples_leaf=10, random_state=42, n_jobs=-1))
    ])
    
    y_train_log = np.log1p(y_train)
    model.fit(X_train, y_train_log)
    print("Model trained successfully")

    print("\nrf_model coefficient")

    feature_names = model.named_steps["preprocessor"].get_feature_names_out()

    feature_importance = pd.DataFrame({
        "Feature": feature_names,
        "Importance": model.named_steps["randomforestregressor"].feature_importances_
    }).sort_values(by="Importance", ascending=False)

    print(feature_importance)

    return model

In [105]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    print("\n" + "=" * 60)
    print("EVALUATING MODEL")
    print("=" * 60)

    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)

    # Predictions in log space
    y_train_pred_log = model.predict(X_train)
    y_test_pred_log  = model.predict(X_test)

    # Reverse transform for real-world metrics
    y_train_pred = np.expm1(y_train_pred_log)
    y_test_pred  = np.expm1(y_test_pred_log)

    # Metrics on original scale
    train_r2   = r2_score(y_train, y_train_pred)
    test_r2    = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse  = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae  = mean_absolute_error(y_train, y_train_pred)
    test_mae   = mean_absolute_error(y_test, y_test_pred)

    train_r2_log = r2_score(y_train_log, y_train_pred_log)
    test_r2_log  = r2_score(y_test_log, y_test_pred_log)

    print("\nTraining set")
    print(f"  R2 (original scale): {train_r2:.4f}")
    print(f"  R2 (log scale):      {train_r2_log:.4f}")
    print(f"  RMSE: {train_rmse:,.2f}  |  MAE: {train_mae:,.2f}")

    print("\nTesting set")
    print(f"  R2 (original scale): {test_r2:.4f}")
    print(f"  R2 (log scale):      {test_r2_log:.4f}")
    print(f"  RMSE: {test_rmse:,.2f}  |  MAE: {test_mae:,.2f}")


    cv_scores = cross_val_score(model, X_train, y_train_log, cv=10, scoring="r2")
    print("\nCross Validation (5-folds) — log scale")
    print(f"  R2 scores: {cv_scores}")
    print(f"  Mean: {cv_scores.mean():.4f}  |  STD: {cv_scores.std():.4f}")

    metrics = {
        "train_r2": train_r2,         "test_r2": test_r2,
        "train_r2_log": train_r2_log, "test_r2_log": test_r2_log,
        "train_rmse": train_rmse,     "test_rmse": test_rmse,
        "train_mae": train_mae,       "test_mae": test_mae,
        "y_train_pred": y_train_pred, "y_test_pred": y_test_pred,
        "cv_scores": cv_scores
    }
    return metrics

In [106]:
# def evaluate_model(model, X_train, X_test, y_train, y_test):
#     print("\n" + "=" * 60)
#     print("EVALUATING MODEL")
#     print("=" * 60)

#     y_test_log  = np.log1p(y_test)
#     y_train_log = np.log1p(y_train)

#     #make prediction 
#     y_train_pred_log = model.predict(X_train)
#     y_train_pred = np.expm1(y_train_pred_log)
    
#     y_test_pred_log = model.predict(X_test)
#     y_test_pred = np.expm1(y_test_pred_log)

#     # evaluate
#     train_r2 = r2_score(y_train, y_train_pred)
#     test_r2 = r2_score(y_test, y_test_pred)

#     train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
#     test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

#     train_mse = (mean_squared_error(y_train, y_train_pred))
#     test_mse = (mean_squared_error(y_test, y_test_pred))
    
#     train_mae = mean_absolute_error(y_train, y_train_pred)
#     test_mae = mean_absolute_error(y_test, y_test_pred)

#     print("\n" + "=" * 60)
#     print("MODEL PERFORMANCE")
#     print("=" * 60)
#     print("\ntTraining set")
#     print(f" R2 score: {train_r2:.4}")
#     print(f" RMSE: {train_rmse:.2}")
#     print(f" MSE: {train_mse:.2}")
#     print(f" MAE: {train_mae:.2}")
#     print("\ntTesting set")
#     print(f" R2 score: {test_r2:.4}")
#     print(f" RMSE: {test_rmse:.2}")
#     print(f" MSE: {test_mse:.2}")
#     print(f" MAE: {test_mae:.2}")

#     # cross validation
#     cv_scores = cross_val_score(model, X_train, y_train_log, cv=5, scoring="r2")
#     print("\nCross Validation (5-folds)")
#     print(f" R2 scores: {cv_scores}")
#     print(f" Cross validation mean: {cv_scores.mean():.4f}")
#     print(f" Cross validation STD: {cv_scores.std():.4f}")

#     metrics = {
#         "train_r2": train_r2,
#         "test_r2": test_r2,
#         "train_rmse": train_rmse,
#         "test_rmse": test_rmse,
#         "train_mse": train_mse,
#         "test_mse": test_mse,
#         "train_mae": train_mae,
#         "test_mae": test_mae,
#         "y_train_pred": y_train_pred,
#         "y_test_pred": y_test_pred,
#         "cv_scores": cv_scores
#     }

#     return metrics

In [107]:
def save_model_artifact(model):
    print("\n" + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(model, "../models/rf_model_pipeline.pkl")
    print("Pipeline model saved successfully")

    print("\n" + "=" * 60)
    print("ALL MODEL ARTIFACT SAVED SUCCESSFULLY")
    print("=" * 60)

In [108]:
def main():
    filepath = "../data/cleaned/cleaned_nigeria_medical_insurance.csv"

    df = load_and_explore_dataset(filepath)

    X, y, numerical_features, categorical_features, feature_columns = features_data(df)

    X_train, X_test, y_train, y_test = split_data(X, y)

    model = build_and_train_model(X_train, y_train, numerical_features, categorical_features)

    mertics = evaluate_model(model, X_train, X_test, y_train, y_test)

    # save_model_artifact(model)

    


if __name__ == "__main__":
    main()

LOAD AND  EXPLORE DATASET
Shape of the dataset
(554, 7)

Check for missing values
age              111
gender           141
bmi                0
children          72
smoker             0
state              0
hospital_bill      0
dtype: int64

First five rows:
    age  gender   bmi  children smoker   state  hospital_bill
0  33.0  Female  22.7       0.0     No  Kaduna    32976705.91
1  64.0  Female  39.0       0.0    Yes   Lagos    24127691.00
2  34.0    Male  33.2       1.0    Yes   Abuja     8392268.25
3  62.0  Female  39.2       NaN    Yes   Lagos    20206206.60
4   NaN    Male  29.7       0.0    Yes    Kano     6535565.00

Descriptive stats
              age         bmi    children  hospital_bill
count  443.000000  554.000000  482.000000   5.540000e+02
mean    38.812641   30.206859    1.338174   1.387578e+07
std     14.236325    5.994047    1.372342   9.057740e+06
min     18.000000   16.000000    0.000000   1.697260e+06
25%     26.000000   25.700000    0.000000   6.557358e+06
50%    